# 📋 Notebook 01: Data Quality Checklist for MMM

**Before you build a Marketing Mix Model, validate your data.** Bad data in = bad insights out.

This notebook runs **10 automated checks** on your marketing dataset — the same categories used by production MMM platforms. By the end, you’ll have a clear pass/fail report and know exactly what to fix.

| Check | What It Catches |
|-------|----------------|
| 1. Schema | Missing required columns (date, KPI, media) |
| 2. Date Frequency | Irregular gaps, mixed cadences |
| 3. Missing Values | Nulls, NaNs, hidden gaps |
| 4. Outliers | Extreme spend or revenue spikes |
| 5. Multicollinearity | Channels that move together (VIF) |
| 6. Zero Variance | Channels with no variation |
| 7. Negative Values | Impossible spend/revenue |
| 8. Data Volume | Enough rows for stable estimation |
| 9. Spend Coverage | Channels with too many zero-spend weeks |
| 10. Target Leakage | Features that suspiciously predict revenue |

---

## Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# --- Chart style (consistent across all notebooks) ---
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'DejaVu Sans'],
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': '#FAFBFC',
    'axes.facecolor': '#FAFBFC',
    'axes.edgecolor': '#D0D7DE',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.color': '#D0D7DE',
    'legend.framealpha': 0.9,
    'legend.edgecolor': '#D0D7DE',
})
COLORS = ['#2563EB', '#F97316', '#10B981', '#EF4444', '#8B5CF6', '#EC4899']

print('✅ Setup complete')

✅ Setup complete


## Load Your Data

We'll use the sample dataset included with these notebooks. **Replace the path below with your own CSV** to validate your data.

In [2]:
df = pd.read_csv('data/sample_mmm_weekly.csv', parse_dates=['date'])

# --- Configure your columns ---
DATE_COL = 'date'
TARGET_COL = 'revenue'
MEDIA_COLS = ['tv_spend', 'facebook_spend', 'google_search_spend', 'radio_spend', 'print_spend']
CONTROL_COLS = ['competitor_spend', 'temperature']

print(f'Dataset: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'Date range: {df[DATE_COL].min().date()} to {df[DATE_COL].max().date()}')
print(f'Media channels: {len(MEDIA_COLS)}')
df.head()

Dataset: 104 rows x 11 columns
Date range: 2023-01-02 to 2024-12-23
Media channels: 5


,date,revenue,tv_spend,facebook_spend,google_search_spend,radio_spend,print_spend,competitor_spend,temperature,black_friday,christmas
0,2023-01-02,415347.38,44297.59,12324.66,21047.02,0.00,8829.08,19018.51,5.5,0,0
1,2023-01-09,448657.78,34361.58,16350.69,57286.45,8007.27,0.00,30129.15,7.4,0,0
2,2023-01-16,446594.40,47055.14,34306.60,21402.59,10756.73,0.00,27026.35,7.1,0,0
3,2023-01-23,451778.48,66783.54,14578.29,25353.44,4977.10,3125.23,38330.92,4.8,0,0
4,2023-01-30,451489.81,33068.57,15195.81,24009.40,0.00,0.00,26243.63,9.1,0,0


## Quick Data Overview

Before diving into checks, let's visualize the data to build intuition.

In [3]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [1, 1.2]})

# Revenue over time
ax = axes[0]
ax.plot(df[DATE_COL], df[TARGET_COL] / 1000, color=COLORS[0], linewidth=2)
ax.fill_between(df[DATE_COL], df[TARGET_COL] / 1000, alpha=0.1, color=COLORS[0])
ax.set_ylabel('Revenue ($K)')
ax.set_title('Revenue Over Time')

# Stacked media spend
ax = axes[1]
bottom = np.zeros(len(df))
for i, col in enumerate(MEDIA_COLS):
    ax.fill_between(df[DATE_COL], bottom, bottom + df[col] / 1000,
                    alpha=0.7, color=COLORS[i % len(COLORS)],
                    label=col.replace('_spend', '').replace('_', ' ').title())
    bottom += df[col].values / 1000
ax.set_ylabel('Media Spend ($K)')
ax.set_title('Media Spend by Channel')
ax.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.savefig('images/01_data_overview.png', dpi=180, bbox_inches='tight')
plt.show()
print('\n✅ Data overview chart saved')


✅ Data overview chart saved


---

## Check 1: Schema Validation

Does the dataset have all required columns? A valid MMM dataset needs:
- **One date column** (parseable as datetime)
- **One target/KPI column** (numeric)
- **At least 2 media spend columns** (numeric, non-negative)

In [4]:
def check_schema(df, date_col, target_col, media_cols):
    issues = []
    
    # Date column
    if date_col not in df.columns:
        issues.append(f'❌ Date column "{date_col}" not found')
    elif not pd.api.types.is_datetime64_any_dtype(df[date_col]):
        issues.append(f'⚠️ Date column "{date_col}" is not datetime type')
    
    # Target column
    if target_col not in df.columns:
        issues.append(f'❌ Target column "{target_col}" not found')
    elif not pd.api.types.is_numeric_dtype(df[target_col]):
        issues.append(f'❌ Target column "{target_col}" is not numeric')
    
    # Media columns
    missing = [c for c in media_cols if c not in df.columns]
    if missing:
        issues.append(f'❌ Missing media columns: {missing}')
    if len(media_cols) < 2:
        issues.append(f'⚠️ Only {len(media_cols)} media channel(s) — need at least 2 for meaningful MMM')
    
    if not issues:
        print('✅ Check 1 PASSED: Schema is valid')
        print(f'   • {len(media_cols)} media channels, 1 target, 1 date column')
    else:
        print('❌ Check 1 FAILED: Schema issues found')
        for issue in issues:
            print(f'   {issue}')
    return len(issues) == 0

check_schema(df, DATE_COL, TARGET_COL, MEDIA_COLS)

✅ Check 1 PASSED: Schema is valid
   • 5 media channels, 1 target, 1 date column


True

## Check 2: Date Frequency

MMM needs **regular time intervals**. Mixed frequencies (some weeks, some days) break the adstock convolution. We check that the gap between consecutive dates is consistent.

In [5]:
def check_date_frequency(df, date_col):
    dates = df[date_col].sort_values()
    gaps = dates.diff().dropna()
    
    # Detect cadence
    median_gap = gaps.median().days
    if median_gap <= 1:
        cadence = 'daily'
    elif 6 <= median_gap <= 8:
        cadence = 'weekly'
    elif 28 <= median_gap <= 31:
        cadence = 'monthly'
    else:
        cadence = 'irregular'
    
    # Check consistency
    gap_days = gaps.dt.days
    irregular = gap_days[gap_days != gap_days.mode()[0]]
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.bar(range(len(gap_days)), gap_days, color=COLORS[0], alpha=0.7, width=1.0)
    ax.axhline(y=gap_days.mode()[0], color=COLORS[3], linestyle='--', label=f'Expected: {gap_days.mode()[0]} days')
    if len(irregular) > 0:
        ax.bar(irregular.index - 1, irregular.values, color=COLORS[3], alpha=0.9, width=1.0)
    ax.set_xlabel('Row Index')
    ax.set_ylabel('Days Between Rows')
    ax.set_title('Date Gap Consistency')
    ax.legend()
    plt.tight_layout()
    plt.savefig('images/01_date_gaps.png', dpi=180, bbox_inches='tight')
    plt.show()
    
    if len(irregular) == 0:
        print(f'✅ Check 2 PASSED: Consistent {cadence} cadence ({gap_days.mode()[0]} days between rows)')
    else:
        print(f'⚠️ Check 2 WARNING: Detected {cadence} cadence but {len(irregular)} irregular gap(s)')
        print(f'   Irregular rows: {irregular.index.tolist()[:10]}')
    return len(irregular) == 0

check_date_frequency(df, DATE_COL)

✅ Check 2 PASSED: Consistent weekly cadence (7 days between rows)


True

## Check 3: Missing Values

PyMC models fail silently with NaN values. Let's check every column.

In [6]:
def check_missing_values(df, media_cols, target_col, control_cols):
    all_cols = [target_col] + media_cols + control_cols
    missing = df[all_cols].isnull().sum()
    missing_pct = (missing / len(df) * 100).round(1)
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 4))
    colors_bar = [COLORS[3] if v > 0 else COLORS[2] for v in missing.values]
    bars = ax.barh(range(len(all_cols)), missing_pct.values, color=colors_bar, alpha=0.8)
    ax.set_yticks(range(len(all_cols)))
    ax.set_yticklabels([c.replace('_', ' ').title() for c in all_cols])
    ax.set_xlabel('Missing %')
    ax.set_title('Missing Values by Column')
    for i, (v, pct) in enumerate(zip(missing.values, missing_pct.values)):
        ax.text(pct + 0.3, i, f'{v} ({pct}%)', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('images/01_missing_values.png', dpi=180, bbox_inches='tight')
    plt.show()
    
    total_missing = missing.sum()
    if total_missing == 0:
        print('✅ Check 3 PASSED: No missing values')
    else:
        print(f'❌ Check 3 FAILED: {total_missing} missing values found')
        for col in all_cols:
            if missing[col] > 0:
                print(f'   • {col}: {missing[col]} missing ({missing_pct[col]}%)')
    return total_missing == 0

check_missing_values(df, MEDIA_COLS, TARGET_COL, CONTROL_COLS)

✅ Check 3 PASSED: No missing values


np.True_

## Check 4: Outlier Detection

Extreme values can dominate Bayesian estimation. We flag values beyond **3x the interquartile range (IQR)** from Q1/Q3.

In [7]:
def check_outliers(df, media_cols, target_col, iqr_multiplier=3.0):
    all_cols = [target_col] + media_cols
    outlier_counts = {}
    
    fig, axes = plt.subplots(1, len(all_cols), figsize=(3 * len(all_cols), 4))
    if len(all_cols) == 1:
        axes = [axes]
    
    for i, col in enumerate(all_cols):
        vals = df[col].dropna()
        q1, q3 = vals.quantile(0.25), vals.quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - iqr_multiplier * iqr, q3 + iqr_multiplier * iqr
        outliers = vals[(vals < lower) | (vals > upper)]
        outlier_counts[col] = len(outliers)
        
        bp = axes[i].boxplot(vals, patch_artist=True,
                            boxprops=dict(facecolor=COLORS[i % len(COLORS)], alpha=0.5),
                            medianprops=dict(color='black', linewidth=2),
                            flierprops=dict(marker='o', markerfacecolor=COLORS[3], markersize=4))
        axes[i].set_title(col.replace('_', '\n').title(), fontsize=9)
        if len(outliers) > 0:
            axes[i].set_title(f'{col.replace("_", chr(10)).title()}\n({len(outliers)} outliers)',
                             fontsize=9, color=COLORS[3])
    
    plt.suptitle(f'Outlier Detection ({iqr_multiplier}x IQR Rule)', fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('images/01_outliers.png', dpi=180, bbox_inches='tight')
    plt.show()
    
    total = sum(outlier_counts.values())
    if total == 0:
        print(f'✅ Check 4 PASSED: No outliers detected ({iqr_multiplier}x IQR)')
    else:
        print(f'⚠️ Check 4 WARNING: {total} outlier(s) detected')
        for col, count in outlier_counts.items():
            if count > 0:
                print(f'   • {col}: {count} outlier(s)')
    return total == 0

check_outliers(df, MEDIA_COLS, TARGET_COL)

⚠️ Check 4 WARNING: 4 outlier(s) detected
   • facebook_spend: 2 outlier(s)
   • google_search_spend: 1 outlier(s)
   • print_spend: 1 outlier(s)


False

## Check 5: Multicollinearity (VIF)

When media channels are highly correlated, the model can't distinguish their individual effects. **Variance Inflation Factor (VIF) > 10** signals a problem; **VIF > 5** is a warning.

> In practice, multicollinearity above 0.9 correlation is the threshold used by production MMM platforms.

In [8]:
from numpy.linalg import inv

def check_multicollinearity(df, media_cols, vif_threshold=10, corr_threshold=0.9):
    media_df = df[media_cols].dropna()
    
    # Correlation matrix
    corr = media_df.corr()
    
    # VIF calculation
    X = media_df.values
    X_std = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-10)
    try:
        corr_matrix = np.corrcoef(X_std, rowvar=False)
        vif = np.diag(inv(corr_matrix + np.eye(len(media_cols)) * 1e-8))
    except:
        vif = np.ones(len(media_cols)) * np.nan
    
    # Visualize correlation heatmap
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Heatmap
    im = ax1.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    labels = [c.replace('_spend', '').replace('_', ' ').title() for c in media_cols]
    ax1.set_xticks(range(len(media_cols)))
    ax1.set_yticks(range(len(media_cols)))
    ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
    ax1.set_yticklabels(labels, fontsize=9)
    for i in range(len(media_cols)):
        for j in range(len(media_cols)):
            ax1.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if abs(corr.values[i,j]) > 0.6 else 'black')
    ax1.set_title('Channel Correlation Matrix')
    plt.colorbar(im, ax=ax1, shrink=0.8)
    
    # VIF bars
    bar_colors = [COLORS[3] if v > vif_threshold else COLORS[1] if v > 5 else COLORS[2] for v in vif]
    ax2.barh(range(len(media_cols)), vif, color=bar_colors, alpha=0.8)
    ax2.axvline(x=vif_threshold, color=COLORS[3], linestyle='--', label=f'Threshold (VIF={vif_threshold})')
    ax2.axvline(x=5, color=COLORS[1], linestyle='--', alpha=0.5, label='Warning (VIF=5)')
    ax2.set_yticks(range(len(media_cols)))
    ax2.set_yticklabels(labels)
    ax2.set_xlabel('Variance Inflation Factor')
    ax2.set_title('VIF by Channel')
    ax2.legend(fontsize=9)
    for i, v in enumerate(vif):
        ax2.text(v + 0.1, i, f'{v:.1f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('images/01_multicollinearity.png', dpi=180, bbox_inches='tight')
    plt.show()
    
    # High correlations
    high_corr = []
    for i in range(len(media_cols)):
        for j in range(i+1, len(media_cols)):
            if abs(corr.values[i,j]) > corr_threshold:
                high_corr.append((media_cols[i], media_cols[j], corr.values[i,j]))
    
    issues = [v > vif_threshold for v in vif]
    if not any(issues) and not high_corr:
        print('✅ Check 5 PASSED: No multicollinearity issues')
    else:
        if any(issues):
            print(f'❌ Check 5 FAILED: High VIF detected')
            for col, v in zip(media_cols, vif):
                if v > vif_threshold:
                    print(f'   • {col}: VIF={v:.1f}')
        if high_corr:
            print(f'⚠️ High correlations (>{corr_threshold}):')
            for c1, c2, r in high_corr:
                print(f'   • {c1} ↔ {c2}: r={r:.3f}')
    return not any(issues)

check_multicollinearity(df, MEDIA_COLS)

✅ Check 5 PASSED: No multicollinearity issues


True

## Check 6: Zero Variance

A channel with constant spend (or all zeros) provides no information for modeling.

In [9]:
def check_zero_variance(df, media_cols):
    issues = []
    for col in media_cols:
        cv = df[col].std() / (df[col].mean() + 1e-10)
        if df[col].std() == 0:
            issues.append(f'❌ {col}: Zero variance (constant value)')
        elif cv < 0.05:
            issues.append(f'⚠️ {col}: Very low variation (CV={cv:.3f})')
    
    if not issues:
        print('✅ Check 6 PASSED: All channels have meaningful variation')
    else:
        print('⚠️ Check 6 WARNING:')
        for issue in issues:
            print(f'   {issue}')
    return len(issues) == 0

check_zero_variance(df, MEDIA_COLS)

✅ Check 6 PASSED: All channels have meaningful variation


True

## Check 7: Negative Values

Media spend and revenue should never be negative. Negative values usually indicate data processing errors (e.g., refunds mixed with revenue).

In [10]:
def check_negative_values(df, media_cols, target_col):
    issues = []
    for col in [target_col] + media_cols:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            issues.append(f'❌ {col}: {neg_count} negative value(s) (min={df[col].min():.2f})')
    
    if not issues:
        print('✅ Check 7 PASSED: No negative values in spend or revenue')
    else:
        print('❌ Check 7 FAILED:')
        for issue in issues:
            print(f'   {issue}')
    return len(issues) == 0

check_negative_values(df, MEDIA_COLS, TARGET_COL)

✅ Check 7 PASSED: No negative values in spend or revenue


True

## Check 8: Data Volume

Bayesian MMM with 5+ channels typically needs **at least 52 rows** (1 year of weekly data). Fewer rows means wider posteriors and less reliable estimates.

Rule of thumb: **10–15 rows per parameter** being estimated.

In [11]:
def check_data_volume(df, media_cols, min_rows=52):
    n_rows = len(df)
    # Rough parameter count: intercept + media channels + adstock + saturation + control + seasonality
    est_params = 1 + len(media_cols) * 3 + len(CONTROL_COLS) + 4  # rough
    rows_per_param = n_rows / est_params
    
    print(f'   Rows: {n_rows}')
    print(f'   Estimated parameters: ~{est_params}')
    print(f'   Rows per parameter: {rows_per_param:.1f}')
    
    if n_rows >= min_rows and rows_per_param >= 5:
        print(f'✅ Check 8 PASSED: Sufficient data ({n_rows} rows, {rows_per_param:.0f} rows/param)')
        return True
    elif n_rows < min_rows:
        print(f'❌ Check 8 FAILED: Only {n_rows} rows (need ≥{min_rows})')
        return False
    else:
        print(f'⚠️ Check 8 WARNING: Low rows-per-parameter ratio ({rows_per_param:.1f})')
        return False

check_data_volume(df, MEDIA_COLS)

   Rows: 104
   Estimated parameters: ~22
   Rows per parameter: 4.7
⚠️ Check 8 WARNING: Low rows-per-parameter ratio (4.7)


False

## Check 9: Spend Coverage

Channels with too many zero-spend weeks are hard to model — there's not enough signal. Flag channels where **>50% of rows are zero**.

In [12]:
def check_spend_coverage(df, media_cols, zero_threshold=0.5):
    fig, ax = plt.subplots(figsize=(10, 4))
    
    labels = []
    zero_pcts = []
    colors_bar = []
    
    for i, col in enumerate(media_cols):
        zero_pct = (df[col] == 0).sum() / len(df)
        zero_pcts.append(zero_pct * 100)
        labels.append(col.replace('_spend', '').replace('_', ' ').title())
        colors_bar.append(COLORS[3] if zero_pct > zero_threshold else COLORS[1] if zero_pct > 0.3 else COLORS[2])
    
    ax.barh(range(len(media_cols)), zero_pcts, color=colors_bar, alpha=0.8)
    ax.axvline(x=zero_threshold * 100, color=COLORS[3], linestyle='--',
              label=f'Threshold ({zero_threshold*100:.0f}%)')
    ax.set_yticks(range(len(media_cols)))
    ax.set_yticklabels(labels)
    ax.set_xlabel('% Weeks with Zero Spend')
    ax.set_title('Spend Coverage by Channel')
    ax.legend()
    for i, pct in enumerate(zero_pcts):
        ax.text(pct + 0.5, i, f'{pct:.0f}%', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig('images/01_spend_coverage.png', dpi=180, bbox_inches='tight')
    plt.show()
    
    sparse_channels = [col for col, pct in zip(media_cols, zero_pcts) if pct > zero_threshold * 100]
    if not sparse_channels:
        print('✅ Check 9 PASSED: All channels have adequate spend coverage')
    else:
        print(f'⚠️ Check 9 WARNING: {len(sparse_channels)} channel(s) with >50% zero-spend weeks')
        for ch in sparse_channels:
            print(f'   • {ch}')
    return len(sparse_channels) == 0

check_spend_coverage(df, MEDIA_COLS)

✅ Check 9 PASSED: All channels have adequate spend coverage


True

## Check 10: Target Leakage

If a feature correlates **too perfectly** with revenue (r > 0.95), it might contain future information (leakage) rather than being a true driver. This can happen when revenue-derived metrics (e.g., ROAS, conversions) are accidentally included as inputs.

In [13]:
def check_leakage(df, media_cols, target_col, control_cols, threshold=0.95):
    all_features = media_cols + control_cols
    correlations = {col: df[col].corr(df[target_col]) for col in all_features}
    
    suspects = {k: v for k, v in correlations.items() if abs(v) > threshold}
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 4))
    labels = [c.replace('_spend', '').replace('_', ' ').title() for c in all_features]
    corr_vals = [correlations[c] for c in all_features]
    bar_colors = [COLORS[3] if abs(v) > threshold else COLORS[0] for v in corr_vals]
    
    ax.barh(range(len(all_features)), corr_vals, color=bar_colors, alpha=0.8)
    ax.axvline(x=threshold, color=COLORS[3], linestyle='--', alpha=0.7)
    ax.axvline(x=-threshold, color=COLORS[3], linestyle='--', alpha=0.7, label=f'Leakage zone (|r|>{threshold})')
    ax.set_yticks(range(len(all_features)))
    ax.set_yticklabels(labels)
    ax.set_xlabel(f'Correlation with {target_col}')
    ax.set_title('Feature-Target Correlations')
    ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.savefig('images/01_leakage_check.png', dpi=180, bbox_inches='tight')
    plt.show()
    
    if not suspects:
        print('✅ Check 10 PASSED: No target leakage detected')
    else:
        print(f'❌ Check 10 FAILED: Possible leakage in {len(suspects)} feature(s)')
        for col, r in suspects.items():
            print(f'   • {col}: r={r:.3f} (suspiciously high)')
    return len(suspects) == 0

check_leakage(df, MEDIA_COLS, TARGET_COL, CONTROL_COLS)

✅ Check 10 PASSED: No target leakage detected


True

---

## Summary Report

Let's run all 10 checks and produce a final scorecard.

In [14]:
print('=' * 60)
print('      DATA QUALITY REPORT')
print('=' * 60)

checks = [
    ('Schema Validation', check_schema(df, DATE_COL, TARGET_COL, MEDIA_COLS)),
    ('Date Frequency', check_date_frequency(df, DATE_COL)),
    ('Missing Values', check_missing_values(df, MEDIA_COLS, TARGET_COL, CONTROL_COLS)),
    ('Outlier Detection', check_outliers(df, MEDIA_COLS, TARGET_COL)),
    ('Multicollinearity', check_multicollinearity(df, MEDIA_COLS)),
    ('Zero Variance', check_zero_variance(df, MEDIA_COLS)),
    ('Negative Values', check_negative_values(df, MEDIA_COLS, TARGET_COL)),
    ('Data Volume', check_data_volume(df, MEDIA_COLS)),
    ('Spend Coverage', check_spend_coverage(df, MEDIA_COLS)),
    ('Target Leakage', check_leakage(df, MEDIA_COLS, TARGET_COL, CONTROL_COLS)),
]

print('\n' + '=' * 60)
print('      SCORECARD')
print('=' * 60)
passed = sum(1 for _, v in checks if v)
for name, result in checks:
    icon = '✅' if result else '❌'
    print(f'  {icon}  {name}')
print(f'\n  Score: {passed}/{len(checks)} checks passed')

if passed == len(checks):
    print('\n  🎉 Your data is ready for modeling!')
elif passed >= 7:
    print('\n  ⚠️  Minor issues — review warnings before proceeding.')
else:
    print('\n  🛑 Significant issues — fix before building your model.')

      DATA QUALITY REPORT
✅ Check 1 PASSED: Schema is valid
   • 5 media channels, 1 target, 1 date column


✅ Check 2 PASSED: Consistent weekly cadence (7 days between rows)


✅ Check 3 PASSED: No missing values


⚠️ Check 4 WARNING: 4 outlier(s) detected
   • facebook_spend: 2 outlier(s)
   • google_search_spend: 1 outlier(s)
   • print_spend: 1 outlier(s)


✅ Check 5 PASSED: No multicollinearity issues
✅ Check 6 PASSED: All channels have meaningful variation
✅ Check 7 PASSED: No negative values in spend or revenue
   Rows: 104
   Estimated parameters: ~22
   Rows per parameter: 4.7
⚠️ Check 8 WARNING: Low rows-per-parameter ratio (4.7)


✅ Check 9 PASSED: All channels have adequate spend coverage


✅ Check 10 PASSED: No target leakage detected

      SCORECARD
  ✅  Schema Validation
  ✅  Date Frequency
  ✅  Missing Values
  ❌  Outlier Detection
  ✅  Multicollinearity
  ✅  Zero Variance
  ✅  Negative Values
  ❌  Data Volume
  ✅  Spend Coverage
  ✅  Target Leakage

  Score: 8/10 checks passed

  ⚠️  Minor issues — review warnings before proceeding.


---

## Next Steps

Now that your data is validated:

- **[Notebook 02: Smart Priors](02-smart-priors-from-data.ipynb)** — Auto-generate informed priors from your data
- **[Notebook 08: Tanh Saturation](08-tanh-saturation-deep-dive.ipynb)** — Understand the saturation function your model will use
- **[PyMC-Marketing Quickstart](https://www.pymc-marketing.io/en/latest/notebooks/mmm/mmm_quickstart.html)** — Fit your first MMM model

**Core concepts:**
- [Data Validation](../docs/data/data-validation.md) — Full reference for all validation categories
- [Data Requirements](../docs/data/data-requirements.md) — What data you need for MMM